In [167]:
import pandas as pd
import sqlite3
from pprint import pprint
from pathlib import Path
import re
import textstat

# caminho para o banco de dados
db_path = '../data/contos.sqlite'
conn = sqlite3.connect(db_path)

# Primeira sessão de análise de dados

In [20]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
display(tables)

,name
0,tales
1,sqlite_sequence
2,classificacoes
3,conto_classificacao


In [21]:
df_tales = pd.read_sql_query("SELECT * FROM tales", conn)

display(df_tales.head(3))

,id,titulo,origem,url,texto_completo
0,1,The Birth of Aistulf,Jacob and Wilhelm Grimm,https://sites.pitt.edu/~dash/manykids.html,The following legend is told about King Aistul...
1,2,As Many Children as There Are Days in the Year,Jacob and Wilhelm Grimm,https://sites.pitt.edu/~dash/manykids.html,Loosduynen (Leusden) is a small village one mi...
2,3,The Woman with Three Hundred and Sixty-Six \nC...,Netherlands,https://sites.pitt.edu/~dash/manykids.html,"So, because the forests of oak, and beech, and..."


In [22]:
len(df_tales)

1683

In [152]:
df = pd.read_csv(Path.cwd().parent / 'data' / '20260114_1630contos_semi_limpos.csv')
print(f"{df.shape=}\n")
display(df.sample(2))

df.shape=(1403, 6)



,Unnamed: 0,url,title,text_EN,region,author
1164,1164,https://sites.pitt.edu/~dash/type1030.html,Saint John and the Devil,"On the other hand, the devil clumsily took hol...",Italy/Austria,NaN
1131,1131,https://sites.pitt.edu/~dash/type1342.html,The Peasant and the Student,The peasant then invited him to stay for dinne...,Germany,NaN


In [153]:
# 1. Rename column
df.rename(columns={'text_EN': 'full_text_EN'}, inplace=True)
# 2. Split logic
def split_source(text):
    if pd.isna(text):
        return None, None
    if 'Source' in text:
        # Split only once at the first occurrence of 'Source'
        parts = text.split('Source', 1)
        return parts[0], 'Source' + parts[1]
    return text, None
# 3. Apply splitting logic to create new columns
df[['clean_text_EN', 'source']] = df['full_text_EN'].apply(
    lambda x: pd.Series(split_source(x))
)

In [162]:
import pandas as pd
import re

# 1. Renomear a coluna original
df.rename(columns={'text_EN': 'full_text_EN'}, inplace=True)

# 2. Função de limpeza aprimorada
def clean_and_format_text(text):
    if pd.isna(text):
        return None
    
    # Normaliza espaços múltiplos e caracteres especiais (tabs, non-breaking spaces)
    text = re.sub(r'[\t\r\f\v]+', ' ', text)
    text = text.replace('\xa0', ' ')
    
    # CORREÇÃO: Espaço faltando após pontuação (ex: "word.Next" -> "word. Next")
    # Mantive sua lógica mas aprimorada para evitar quebras em abreviações como "U.S.A."
    text = re.sub(r'([.!?])([A-Z])', r'\1 \2', text)
    
    # MELHORIA: Junta palavras hifenizadas que foram quebradas por linha (comum em OCR)
    # Ex: "beauti-\nful" -> "beautiful"
    text = re.sub(r'(\w)-\n\s*(\w)', r'\1\2', text)
    
    # PARAGRAFAÇÃO: Adiciona quebra de linha dupla antes de transições típicas
    # Primeiro "achatamos" as quebras existentes para garantir uniformidade
    text = re.sub(r'\n+', ' ', text)
    transitions = r"(Then|Once|But|After|When|In|The|A|His|Her|They|It|One day|Soon)"
    text = re.sub(r'([.!?])\s+(?=' + transitions + r'|")', r'\1\n\n', text)
    
    # LIMPEZA FINAL: Remove espaços duplos e garante no máximo duas quebras de linha
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

# 3. Lógica de Separação e Processamento
def process_text_and_source(text):
    if pd.isna(text):
        return None, None
        
    # Separa na primeira ocorrência de 'Source'
    if 'Source' in text:
        parts = text.split('Source', 1)
        clean_text = parts[0]
        source_part = ('Source' + parts[1]).strip()
    else:
        clean_text = text
        source_part = None
        
    # Aplica a limpeza avançada apenas no conteúdo principal
    refined_text = clean_and_format_text(clean_text)
    
    return refined_text, source_part

# 4. Aplica a transformação
df[['clean_text_EN', 'source']] = df['full_text_EN'].apply(
    lambda x: pd.Series(process_text_and_source(x))
)

In [168]:
# Garantir que não haja NaN para evitar erros nas funções de texto
df['clean_text_EN'] = df['clean_text_EN'].fillna('')
# 1. Métricas Quantitativas
df['word_count'] = df['clean_text_EN'].apply(lambda x: len(x.split()))
df['reading_time'] = df['word_count'] / 200
# 2. Métricas de Legibilidade (textstat)
# Flesch Reading Ease (0-100, quanto maior mais fácil)
df['flesch_reading_ease'] = df['clean_text_EN'].apply(lambda x: textstat.flesch_reading_ease(x) if x else None)
# Dale-Chall Readability Score (usa vocabulário familiar)
df['dale_chall_readability'] = df['clean_text_EN'].apply(lambda x: textstat.dale_chall_readability_score(x) if x else None)
# Visualizar as novas colunas
display(df[['clean_text_EN', 'word_count', 'reading_time', 'flesch_reading_ease', 'dale_chall_readability']].head())

,clean_text_EN,word_count,reading_time,flesch_reading_ease,dale_chall_readability
0,The following legend is told about King Aistul...,81,0.405,85.947556,6.779279
1,Loosduynen (Leusden) is a small village one mi...,250,1.250,72.300400,9.448694
2,"So, because the forests of oak, and beech, and...",2709,13.545,75.547294,7.532870
3,"In the times of Agelmund, the King of the Lang...",168,0.840,80.508000,7.383411
4,Warin was a count of Altorf and Ravensburg in ...,589,2.945,72.175178,8.155761


In [169]:
display(df.sample(2))

,Unnamed: 0,url,title,full_text_EN,region,author,clean_text_EN,source,word_count,reading_time,flesch_reading_ease,dale_chall_readability
602,602,https://sites.pitt.edu/~dash/type0510a.html,Little Saddleslut,"The mother's spindle fell, and they left her a...",Greece,NaN,"The mother's spindle fell, and they left her a...","Source: Edmund Martin Geldart,Folk-Lore of Mod...",1382,6.91,89.062498,6.550772
948,948,https://sites.pitt.edu/~dash/bald.html,A Man and Two Wives,The Moral'Tis a much Harder Thing to Please Tw...,Greece,Aesop,The Moral'Tis a much Harder Thing to Please Tw...,"Source: Roger L'Estrange,Fables of Æsop and O...",22,0.11,88.368636,8.316336


In [170]:
df.to_csv(Path.cwd().parent / 'data' / '20260115_1403_contos_metadados_basicos.csv')